# 07 — Train the defect-region classifier

## Objective

Train and save a scaled, class-weighted logistic regression classifier using the six RGB features for all labelled training patches from Notebook 06.

## Workflow

Load `region_features.csv`, select RGB inputs and binary labels, fit the pipeline, inspect coefficients and save `defect_region_classifier.joblib` for Notebook 08.

The implemented workflow uses all training patches. It does not perform the group holdout split described in the original introduction, and it does not report holdout metrics here. COCO-derived labels are targets; only RGB statistics enter the model.

### Running this review copy

Use the original project `notebooks` folder as the kernel working directory; this copy is stored in its `for review` subfolder. The existing `../data` and `../models` paths depend on that working directory. Run the workflow in order in a fresh kernel. Stored outputs are retained from the source notebook and were not regenerated during this readability review.

## Setup and file locations

Import dependencies and locate the feature CSV from Notebook 06 and the model output.

In [1]:
# Standard library imports
from pathlib import Path

# Numerical and tabular data processing
import numpy as np
import pandas as pd

# Machine learning utilities
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Visualisation
import matplotlib.pyplot as plt

# Model saving
import joblib


# Use a fixed random seed for reproducibility.
RANDOM_SEED = 67

# Input CSV generated by Notebook 06.
FEATURES_PATH = Path("../data/processed/region_features.csv")

# Output location for the trained region classifier.
MODEL_PATH = Path("../models/defect_region_classifier.joblib")

## Load and Inspect Training Features

This section imports the labelled patch dataset generated in
Notebook 06.

The dataset contains RGB statistics, patch coordinates,
source image identifiers and binary defect labels.

Only RGB statistical features will be used as predictors.
Image identifiers are retained to ensure group-aware
training and holdout separation.

In [2]:
# Load the labelled patch features generated in Notebook 06.
region_df = pd.read_csv(FEATURES_PATH)

# Display the dataset dimensions.
print("Dataset shape:", region_df.shape)

# Display the number of unique source images.
print("Unique images:", region_df["image_id"].nunique())

# Confirm the number of examples belonging to each class.
print("\nPatch label distribution:")
print(region_df["label"].value_counts().sort_index())

# Display the first five rows to verify the imported structure.
display(region_df.head())

Dataset shape: (1300762, 13)
Unique images: 3347

Patch label distribution:
label
0    1235328
1      65434
Name: count, dtype: int64


,image_id,filename,x,y,width,height,mean_r,mean_g,mean_b,std_r,std_g,std_b,label
0,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,0,0,32,32,20.934570,22.108398,21.138672,11.025300,11.159137,11.793400,0
1,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,32,0,32,32,22.056641,22.954102,22.741211,12.000843,12.616676,13.242274,0
2,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,64,0,32,32,24.300781,24.996094,23.770508,10.758053,11.645143,12.337976,0
3,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,96,0,32,32,21.693359,23.418945,22.337891,11.463244,11.984743,11.413700,0
4,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,128,0,32,32,21.645508,23.134766,20.901367,11.060837,11.741328,11.364480,0


## Select features and labels

Use six RGB columns as predictors and the binary patch label as the target. Image IDs and patch coordinates are excluded from inputs. Check for missing features and confirm that both classes are present.

In [3]:
# Select the six RGB statistical features used by the classifier.
RGB_FEATURE_COLUMNS = [
    "mean_r",
    "mean_g",
    "mean_b",
    "std_r",
    "std_g",
    "std_b"
]

# Extract the six RGB features for every training patch.
X_train = region_df[RGB_FEATURE_COLUMNS]

# Extract the COCO-derived defect labels.
# 0 = NORMAL, 1 = DEFECT.
y_train = region_df["label"]

# Verify that the training data contains no missing values.
assert not X_train.isna().any().any(), (
    "Missing RGB feature values detected."
)

# Verify that only the expected classes are present.
assert set(y_train.unique()) == {0, 1}, (
    "Unexpected class labels detected."
)

# Display the dimensions of the training dataset.
print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

# Display class distribution.
print("\nClass distribution:")
print(y_train.value_counts().sort_index())

Training features: (1300762, 6)
Training labels: (1300762,)

Class distribution:
label
0    1235328
1      65434
Name: count, dtype: int64


## Preview the pipeline

This cell displays an unfitted example pipeline. The named pipeline below is the one fitted and saved; this preview does not train a second model.

In [4]:
Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        random_state=67,
        max_iter=1000
    ))
])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",67
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and 

## Train Logistic Regression Classifier

This section trains a supervised logistic regression model to
classify individual image patches as NORMAL (0) or DEFECT (1).

The model uses six RGB statistical features extracted in Notebook 06.

A StandardScaler is used to standardise the numerical features,
while balanced class weighting compensates for the unequal numbers
of defective and non-defective patches.

The entire training dataset is used. Model evaluation will be
performed separately using the existing validation dataset.

In [5]:
# Create a machine learning pipeline containing two stages:
# 1. Standardise the six RGB input features.
# 2. Train a logistic regression classifier.

region_model = Pipeline([

    # Standardise each feature to have approximately
    # zero mean and unit standard deviation.
    ("scaler", StandardScaler()),

    # Train a logistic regression classifier.
    ("classifier", LogisticRegression(

        # Increase the weighting of the minority DEFECT class.
        class_weight="balanced",

        # Allow sufficient iterations for convergence.
        max_iter=1000,

        # Use a consistent random seed for reproducibility.
        random_state=RANDOM_SEED
    ))
])

print("Model pipeline created:")
print(region_model)

Model pipeline created:
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=67))])


## Fit the Defect Region Classifier

The logistic regression pipeline is fitted using all labelled
training patches.

The StandardScaler learns feature distributions from the
training data, and logistic regression learns the relationship
between RGB characteristics and COCO-derived defect labels.

No validation images or validation COCO annotations are used.

In [6]:
import time

# Record the starting time to measure training duration.
start_time = time.perf_counter()

# Fit the complete pipeline on the training patches.
# This automatically fits the StandardScaler before
# training the logistic regression classifier.
region_model.fit(X_train, y_train)

# Calculate the elapsed training time.
training_time = time.perf_counter() - start_time

print("Defect region classifier training complete.")
print(f"Training time: {training_time:.2f} seconds")

Defect region classifier training complete.
Training time: 3.49 seconds


## Inspect Model Coefficients

This section examines the coefficients learned by logistic regression.

Because the RGB features were standardised prior to training,
the coefficients provide an indication of their relative
contributions to the classifier's decision function.

Coefficient magnitude indicates the strength of association
within the fitted model, while its sign indicates the direction
of that association.

In [7]:
# Access the trained logistic regression classifier
# from the fitted pipeline.
classifier = region_model.named_steps["classifier"]

# Retrieve the coefficient associated with each RGB feature.
coefficients = classifier.coef_[0]

# Create a DataFrame pairing each feature with its coefficient.
coefficient_df = pd.DataFrame({
    "Feature": RGB_FEATURE_COLUMNS,
    "Coefficient": coefficients
})

# Sort features by the magnitude of their coefficients.
coefficient_df["Absolute coefficient"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    by="Absolute coefficient",
    ascending=False
)

# Display the ordered coefficients.
display(coefficient_df)

,Feature,Coefficient,Absolute coefficient
2,mean_b,4.270743,4.270743
1,mean_g,-2.698777,2.698777
4,std_g,-2.465269,2.465269
5,std_b,1.133834,1.133834
3,std_r,1.016449,1.016449
0,mean_r,0.253506,0.253506


## Save Trained Classifier

The fitted logistic regression pipeline is saved for subsequent
evaluation on the independent validation dataset.

Saving the complete pipeline preserves both the fitted
StandardScaler and the trained logistic regression classifier.

Notebook 08 will load this model to predict defect locations
using RGB features from previously unseen images.

In [8]:
# Ensure the output directory exists.
MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Save the complete trained pipeline.
joblib.dump(region_model, MODEL_PATH)

print(f"Trained classifier saved to: {MODEL_PATH}")

Trained classifier saved to: ..\models\defect_region_classifier.joblib
